# Chapter 1: Common Conventions and API Elements of scikit-learn

## 📋 Summary

This chapter introduces the foundational design philosophy and core API elements of scikit-learn — the most widely used Python machine learning library. It covers how scikit-learn is built around a consistent, modular interface that makes it easy to learn, combine, and extend different components.

The chapter explains the three fundamental types of objects in scikit-learn: **estimators** (which learn from data), **transformers** (which modify data), and **predictors** (which make predictions). All of these follow the same basic interface using `fit()`, `transform()`, and `predict()` methods, enabling seamless interoperability.

**Pipelines** are introduced as a powerful abstraction for chaining together multiple preprocessing steps and models into a single, reproducible workflow. This is critical in production ML engineering where reproducibility, maintainability, and automation (MLOps) matter.

The chapter also covers hyperparameter tuning with `GridSearchCV()` and `RandomizedSearchCV()`, metadata and estimator tags, and best practices for API usage. Understanding these conventions is essential because every subsequent chapter builds upon them.

---

## 🧠 Theoretical Explanation

### The Estimator Interface
At the heart of scikit-learn is the **estimator** concept. Every learning algorithm — from linear regression to neural networks — is an estimator. The interface is minimal by design:
- `fit(X, y)` — learns parameters from training data
- `predict(X)` — generates predictions on new data
- `fit_predict(X)` — shorthand used in unsupervised learning

### Transformers
Transformers modify data rather than predict. They also use `fit()` to learn statistics (e.g., mean, std) from training data and `transform()` to apply the modification. The `fit_transform()` shorthand applies both steps in one call — but should **only** be used on training data. On test data, use only `transform()` to avoid data leakage.

### Pipelines
A `Pipeline` chains transformers and a final estimator into a single object. This ensures:
1. The same preprocessing is applied consistently to train and test data
2. Hyperparameter tuning via `GridSearchCV` can tune preprocessing and model params simultaneously
3. The workflow is reproducible and easy to deploy

### Hyperparameter Tuning
**Hyperparameters** are settings not learned from data (e.g., `n_estimators` in Random Forest). Two main strategies:
- **GridSearchCV**: Exhaustively tries every combination of a parameter grid. Thorough but expensive.
- **RandomizedSearchCV**: Samples a fixed number of combinations randomly. Faster and often equally effective.

### MLOps Context
MLOps applies DevOps principles to ML: continuous training, monitoring, and deployment. scikit-learn supports this via `Pipeline`, `joblib` for model persistence, and compatibility with platforms like MLflow.


## 1.1 Understanding Estimators

Estimators implement `fit()` to learn from data and `predict()` to generate predictions. Here we use `LinearRegression` as the simplest example.

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Example data
X = np.array([[1], [2], [3], [4], [5]])  # Feature matrix
y = np.array([1, 2, 3, 3.5, 5])          # Target values

# Create and fit the model
model = LinearRegression()
model.fit(X, y)

# Predict values for new data
X_new = np.array([[6], [7]])
predictions = model.predict(X_new)
print("Predictions:", predictions)

### fit_predict() with Unsupervised Learning

`fit_predict()` is typically used in unsupervised learning where we want labels for the same data we trained on.

In [ ]:
from sklearn.cluster import KMeans

X = np.array([[1], [2], [3], [4], [5]])

# KMeans clustering with fit_predict
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)
print("Cluster labels:", labels)

## 1.2 Transformers and the transform() Method

`StandardScaler` is a transformer that standardizes features to mean=0 and std=1 (z-scores). We `fit()` on training data only, then `transform()` on both train and test.

In [ ]:
from sklearn.preprocessing import StandardScaler

X = np.array([[1, 2], [3, 4], [5, 6]])

scaler = StandardScaler()

# Fit then transform separately
scaler.fit(X)
X_scaled = scaler.transform(X)
print("Scaled (fit + transform separately):\n", X_scaled)

# Or use fit_transform (only on training data!)
X_scaled2 = scaler.fit_transform(X)
print("\nScaled (fit_transform):\n", X_scaled2)

## 1.3 Custom Estimators and Transformers

scikit-learn allows you to build custom estimators by subclassing `BaseEstimator` and `TransformerMixin`. This makes custom objects compatible with `Pipeline` and `GridSearchCV`.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class CustomScaler(BaseEstimator, TransformerMixin):
    """A custom transformer that clips values to a given range."""
    
    def __init__(self, min_val=0, max_val=1):
        self.min_val = min_val
        self.max_val = max_val
    
    def fit(self, X, y=None):
        # No learning needed for this transformer
        return self
    
    def transform(self, X):
        return np.clip(X, self.min_val, self.max_val)

# Test our custom transformer
X = np.array([[0.5], [1.5], [-0.3], [2.0]])
custom = CustomScaler(min_val=0, max_val=1)
print("Clipped values:", custom.fit_transform(X).flatten())

## 1.4 Pipelines and Workflow Automation

`Pipeline` chains multiple steps. All intermediate steps must be transformers; the last step can be any estimator. This ensures preprocessing is always applied consistently.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Load data
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

# Build pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=200))
])

# Fit and score
pipeline.fit(X_train, y_train)
print("Pipeline accuracy:", pipeline.score(X_test, y_test))

## 1.5 Common Attributes and Methods

scikit-learn models expose learned parameters via attributes like `coef_` and `intercept_`. The `score()` method returns the default evaluation metric (R² for regressors, accuracy for classifiers).

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1, 2, 3, 3.5, 5])

model = LinearRegression()
model.fit(X, y)

print("Coefficients (slope):", model.coef_)
print("Intercept:", model.intercept_)
print("R-squared score:", model.score(X, y))

## 1.6 Hyperparameter Tuning with GridSearchCV

`GridSearchCV` exhaustively searches over a parameter grid using cross-validation to find the best combination.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import load_iris

iris = load_iris()
X, y = iris.data, iris.target

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, None]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy'
)
grid_search.fit(X, y)

print("Best params:", grid_search.best_params_)
print("Best score:", grid_search.best_score_)

### set_params() and get_params()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.set_params(n_estimators=100, max_depth=10, random_state=42)
print(model.get_params())

## 🔑 Key Takeaways

- **Uniform API**: All scikit-learn objects use `fit()`, `transform()`, and/or `predict()` — making code readable and consistent.
- **Estimators** learn from data; **transformers** modify data; **pipelines** chain them together.
- Always `fit()` only on training data to prevent data leakage. Use `transform()` on test data.
- `Pipeline` is the best practice for production-level ML workflows — it ensures reproducibility.
- `GridSearchCV` automates hyperparameter tuning with cross-validation.
- Custom estimators can be created by subclassing `BaseEstimator` and `TransformerMixin`.
